# Loading database

In [2]:
import sqlite3
conn = sqlite3.connect("products.db")
cur = conn.cursor()
cur.execute("PRAGMA table_info(transactions)")
rows = cur.fetchall()
conn.close()
print(rows)

[(0, 'id', 'INTEGER', 0, None, 1), (1, 'product_id', 'INTEGER', 1, None, 0), (2, 'product_name', 'TEXT', 1, None, 0), (3, 'brand', 'TEXT', 1, None, 0), (4, 'category', 'TEXT', 1, None, 0), (5, 'color', 'TEXT', 1, None, 0), (6, 'action', 'TEXT', 1, None, 0), (7, 'qty_delta', 'INTEGER', 0, '0', 0), (8, 'unit_price', 'REAL', 0, None, 0), (9, 'notes', 'TEXT', 0, None, 0), (10, 'ts', 'DATETIME', 0, 'CURRENT_TIMESTAMP', 0)]


In [13]:
schema = "table name: transactions\n" + "\n".join([f"{r[1]} ({r[2]})" for r in rows])
print(schema)

table name: transactions
id (INTEGER)
product_id (INTEGER)
product_name (TEXT)
brand (TEXT)
category (TEXT)
color (TEXT)
action (TEXT)
qty_delta (INTEGER)
unit_price (REAL)
notes (TEXT)
ts (DATETIME)


In [ ]:
# Helper function to get database schema
def get_schema(db_path: str) -> str:
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("PRAGMA table_info(transactions)")
    rows = cur.fetchall()
    conn.close()
    schema = "table name: transactions\n" + "\n".join([f"{r[1]} ({r[2]})" for r in rows])
    return schema
    

# Building the pipeline

## 1. Generate version 1 SQL query

In [ ]:
import os
from dotenv import load_dotenv
import aisuite as ai

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
client = ai.Client()

# Function to generate version 1 SQL
def generate_sql(question: str, schema:str) -> str:
    prompt =  f"""
    You are a SQL assistant. Given the schema and the user's question, write a SQL query for SQLite.

    Schema:
    {schema}

    User question:
    {question}

    Respond with the SQL only.
    """
    # Prompt model to generate SQL
    response = client.chat.completions.create(
        model="openai:gpt-4.1",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=0
    )
    
    return response.choices[0].message.content.strip()

In [6]:
schema = """
Table name: transactions
id (INTEGER)
product_id (INTEGER)
product_name (TEXT)
brand (TEXT)
category (TEXT)
color (TEXT)
action (TEXT)
qty_delta (INTEGER)
unit_price (REAL)
notes (TEXT)
ts (DATETIME)
"""

question = "Which color of product has the highest total sales?"

sql_v1 = generate_sql(question, schema)
print("Version 1 SQL :-")
print(sql_v1)

Version 1 SQL :-
```sql
SELECT color, SUM(qty_delta * unit_price) AS total_sales
FROM transactions
WHERE action = 'sale'
GROUP BY color
ORDER BY total_sales DESC
LIMIT 1;
```


In [ ]:
import pandas as pd
# Helper function to execute sql query
def execute_sql(query:str, db_path:str) -> pd.DataFrame:
    q = query.strip().removeprefix("```sql").removesuffix("```").strip()
    conn = sqlite3.connect(db_path)
    output = pd.read_sql_query(q, conn)
    conn.close()
    return output

In [8]:
df_sql_v1 = execute_sql(sql_v1, db_path="products.db")
df_sql_v1

,color,total_sales
0,blue,-190571.46


## 2. Reflect on the feedback and create version 2 SQL query

In [ ]:
import json
# Function to generate refined SQL query based on external feedback
def refine_sql(question: str, sql_query: str, df_feedback: pd.DataFrame, schema: str) -> tuple[str, str]:
    prompt =  f"""
    You are a SQL reviewer and refiner.

    User asked:
    {question}

    Original SQL:
    {sql_query}

    SQL Output:
    {df_feedback.to_markdown(index=False)}

    Table Schema:
    {schema}

    Step 1: Briefly evaluate if the SQL output answers the user's question.
    Step 2: If the SQL could be improved, provide a refined SQL query.
    If the original SQL is already correct, return it unchanged.

    Return a strict JSON object with two fields:
    - "feedback": brief evaluation and suggestions
    - "refined_sql": the final SQL to run
    """
    # Prompt model to generate refined SQL query
    response = client.chat.completions.create(
        model="openai:gpt-4.1",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=1.0
    )
    content =  response.choices[0].message.content
    
    # Extract feedback and refined_sql from the outputed json
    obj = json.loads(content)
    feedback = str(obj.get("feedback", "")).strip()
    refined_sql = str(obj.get("refined_sql")).strip()
    
    return feedback, refined_sql

In [10]:
feedback, sql_v2 = refine_sql(
    question=question,
    sql_query=sql_v1,
    df_feedback=df_sql_v1,
    schema=schema
)
print(feedback)
print(sql_v2)

The SQL correctly calculates total sales per color using qty_delta * unit_price, returning the color with the highest total sales. However, the negative total_sales value suggests qty_delta may be negative for sales. To ensure total sales are positive, use ABS(qty_delta) when action = 'sale', as sales typically reduce inventory and qty_delta could be negative. The refined SQL clarifies intent and guarantees correct aggregation.
SELECT color, SUM(ABS(qty_delta) * unit_price) AS total_sales
FROM transactions
WHERE action = 'sale'
GROUP BY color
ORDER BY total_sales DESC
LIMIT 1;


In [11]:
df_sql_v2 = execute_sql(sql_v2, db_path="products.db")
df_sql_v2

,color,total_sales
0,white,358315.09


## 3. Creating the end-to-end workflow

In [18]:
def run_sql_workflow(db_path: str, question: str):
    # 1. Get Schema
    schema =  get_schema(db_path)
    
    # 2. Generate Version 1 SQL
    sql_v1 = generate_sql(question, schema)
    print("Version 1 :- SQL")
    print(sql_v1)
    
    # 3. Execute Version 1 SQL
    df_v1 = execute_sql(sql_v1, db_path)
    print("Version 1 :- SQL output")
    print(df_v1)
    
    # 4. Reflect on Version 1 SQL with execution feedback 
    feedback, sql_v2 = refine_sql(question, sql_v1, df_v1, schema)
    print("Version 2 :- Feedback")
    print(feedback)
    print("Version 2 :- SQL")
    print(sql_v2)
    
    # 5. Execute Version 2 SQL
    df_v2 = execute_sql(sql_v2, db_path)
    print("Version 2 :- SQL output")
    print(df_v2)

In [19]:
run_sql_workflow("products.db", "Which color of product has the highest total sales?")

Version 1 :- SQL
```sql
SELECT color, SUM(qty_delta * unit_price) AS total_sales
FROM transactions
WHERE action = 'sale'
GROUP BY color
ORDER BY total_sales DESC
LIMIT 1;
```
Version 1 :- SQL output
  color  total_sales
0  blue   -190571.46
Version 2 :- Feedback
The SQL logic is nearly correct: it calculates total sales by multiplying qty_delta and unit_price, filtered by where action is 'sale', grouped by color, and returns the color with the highest total sales. However, the negative value in total_sales suggests that qty_delta may be negative for sales records. Normally, for sales, you would expect to sum the absolute value of qty_delta or multiply by -1 to show positive revenue (if qty_delta is negative for sales). The query should ensure total sales is reported as a positive value.
Version 2 :- SQL
SELECT color, SUM(ABS(qty_delta) * unit_price) AS total_sales
FROM transactions
WHERE action = 'sale'
GROUP BY color
ORDER BY total_sales DESC
LIMIT 1;
Version 2 :- SQL output
   color 